# Projeto de Ciência de Dados — Metodologia CRISP-DM
**Tema:** Predição de Não Comparecimento (*No-Show*) em Agendamentos Automotivos — **TechMotors (TCC)**  
**Disciplina:** Ciência de Dados e Aprendizagem de Máquina  
**Docente:** Profa. Kadidja Valéria  
**Base de Dados:** Conjunto sintético (fictício) com **1.000 registros** simulando o banco de dados operacional da plataforma TechMotors.

---

### Visão Geral do Ciclo de Vida CRISP-DM
Este notebook segue integralmente as 6 fases do modelo **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*), destacando sua natureza **iterativa** (não linear), a centralidade dos dados e o foco em **impacto real de negócio** acima de métricas estatísticas isoladas:
1. **Entendimento do Negócio**
2. **Entendimento dos Dados**
3. **Preparação dos Dados** *(50% a 70% do esforço do projeto)*
4. **Modelagem** *(com iteração registrada de retorno à Preparação)*
5. **Avaliação** *(Impacto Real > Precisão Técnica)*
6. **Implementação** *(Arquitetura, Exportação do Modelo e Monitoramento)*


---
## Fase 1: Entendimento do Negócio
**Objetivo Central:** Traduzir o problema de negócio em metas técnicas viáveis antes de investir recursos computacionais.

### Checklist de Execução
- **Objetivos de Negócio e Critérios de Sucesso:**
  - *Cenário:* Oficinas mecânicas parceiras da plataforma TechMotors sofrem com o não comparecimento (*no-show*) de clientes agendados, deixando boxes e mecânicos ociosos e gerando perda direta de faturamento.
  - *Meta de Negócio:* Reduzir a taxa de *no-show* em pelo menos **20%** por meio da identificação antecipada de agendamentos de alto risco, permitindo confirmação ativa via WhatsApp ou remanejamento de agenda.
- **Avaliação da Situação (Recursos, Riscos e Custo-Benefício):**
  - *Recursos:* Ambiente Google Colab (Python, Pandas, Scikit-Learn, Matplotlib/Seaborn) e base simulada de 1.000 agendamentos estruturada conforme a modelagem de banco de dados do TCC.
  - *Riscos:* Uso inicial de dados fictícios (exigindo retreinamento quando a plataforma entrar em produção com dados reais) e risco de desconforto caso clientes recebam mensagens excessivas.
  - *Custo-Benefício:* O custo de disparar um lembrete automático extra é praticamente nulo, enquanto evitar uma única hora de box ocioso preserva o faturamento da oficina.
- **Meta Técnica de Mineração de Dados:**
  - Construir e comparar modelos de **classificação supervisionada** para prever a probabilidade de falta (`0 = Compareceu`, `1 = No-Show`), priorizando alto **Recall** na classe `1` (capturar o máximo de faltas reais antes que aconteçam).
- **Plano de Projeto e Matriz de Papéis:**
  - *Patrocinador (Gestor da Oficina):* Atua fortemente nas Fases 1 (Negócio) e 5 (Avaliação).
  - *Equipes Integradas (TI + Negócio):* Atuam nas Fases 2 (Dados) e 6 (Implementação).
  - *Analista de BD / Cientista de Dados:* Concentra o esforço técnico nas Fases 3 (Preparação) e 4 (Modelagem).


---
## Fase 2: Entendimento dos Dados
**Objetivo Central:** Acessar, quantificar e qualificar a matéria-prima disponível para evitar falhas imprevistas na modelagem.

### Checklist de Execução
1. **Coletar dados iniciais:** Geração da base fictícia com **1.000 registros** distribuídos em 3 tabelas relacionais (`clientes`, `veiculos`, `agendamentos`).
2. **Descrever quantidade e qualidade:** Volume, tipos de dados e distribuição das variáveis.
3. **Explorar dados:** Gráficos exploratórios e validação das hipóteses iniciais.
4. **Verificar qualidade:** Identificação proposital de erros de digitação (ex.: veículo ano `2099`, quilometragem negativa `-5000`), dados omissos (nulos) e campos confidenciais (`cpf`).


In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

# Configuração visual dos gráficos e semente de reprodutibilidade
sns.set_theme(style="whitegrid")
np.random.seed(42)
N = 1000

print(f"Ambiente configurado com sucesso! Gerando base sintética de {N} registros...")


In [ ]:
# 1. Coleta Inicial: Geração das 3 tabelas simulando o banco da TechMotors (1.000 registros)
clientes = pd.DataFrame({
    'id_cliente': range(1, N + 1),
    'cpf': [
        f'{np.random.randint(100,999)}.{np.random.randint(100,999)}.{np.random.randint(100,999)}-{np.random.randint(10,99)}'
        for _ in range(N)
    ],
    'idade': np.random.randint(20, 70, size=N),
    'faltas_anteriores': np.random.choice([0, 1, 2, 3], size=N, p=[0.65, 0.20, 0.10, 0.05])
})

veiculos = pd.DataFrame({
    'id_veiculo': range(1001, 1001 + N),
    'id_cliente': range(1, N + 1),
    'ano_veiculo': np.random.randint(2005, 2027, size=N),
    'km_atual': np.random.randint(5000, 190000, size=N)
})

# Injetando erros propositais de digitação para simular problemas reais de qualidade
veiculos.loc[[10, 45, 120, 350, 712, 899], 'ano_veiculo'] = 2099
veiculos.loc[[25, 88, 410, 620, 905], 'km_atual'] = -5000

servicos = ['Revisão Geral', 'Troca de Óleo', 'Freios', 'Suspensão', 'Alinhamento']
dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado']

dias_antecedencia = np.random.randint(1, 22, size=N)
confirmou_msg = np.random.choice([0, 1], size=N, p=[0.35, 0.65])
valor_estimado = np.round(np.random.uniform(150, 2500, size=N), 2)

# Regra comportamental realista para gerar a variável alvo (no_show)
prob_falta = (
    0.08
    + 0.42 * (confirmou_msg == 0)
    + 0.16 * (dias_antecedencia > 10)
    + 0.20 * (clientes['faltas_anteriores'] >= 1)
    - 0.05 * (valor_estimado > 1000)
)
prob_falta = np.clip(prob_falta, 0.02, 0.95)
no_show = np.random.binomial(1, prob_falta)

agendamentos = pd.DataFrame({
    'id_agendamento': range(5001, 5001 + N),
    'id_cliente': range(1, N + 1),
    'tipo_servico': np.random.choice(servicos, size=N),
    'dias_antecedencia': dias_antecedencia,
    'dia_semana': np.random.choice(dias, size=N),
    'valor_estimado': valor_estimado,
    'confirmou_msg': confirmou_msg.astype(float),
    'no_show': no_show
})

# Injetando dados omissos (valores nulos)
agendamentos.loc[np.random.choice(N, 40, replace=False), 'valor_estimado'] = np.nan
agendamentos.loc[np.random.choice(N, 30, replace=False), 'confirmou_msg'] = np.nan

# Opcional: salvar os CSVs brutos no ambiente do Colab para consulta/exportação
clientes.to_csv('techmotors_clientes_bruto.csv', index=False)
veiculos.to_csv('techmotors_veiculos_bruto.csv', index=False)
agendamentos.to_csv('techmotors_agendamentos_bruto.csv', index=False)

print("Tabelas criadas e salvas em CSV:")
print(f"- Clientes:     {clientes.shape}")
print(f"- Veículos:     {veiculos.shape}")
print(f"- Agendamentos: {agendamentos.shape}")
display(agendamentos.head())


In [ ]:
# 2. Verificação de Qualidade dos Dados (Auditoria Inicial)
print("=== AUDITORIA DE QUALIDADE (FASE 2) ===")
print(f"1. Veículos com ano impossível (> 2026):   {(veiculos['ano_veiculo'] > 2026).sum()} registros")
print(f"2. Veículos com quilometragem negativa:    {(veiculos['km_atual'] < 0).sum()} registros")
print(f"3. Valores nulos em 'valor_estimado':      {agendamentos['valor_estimado'].isnull().sum()} registros")
print(f"4. Valores nulos em 'confirmou_msg':       {agendamentos['confirmou_msg'].isnull().sum()} registros")
print(f"5. Presença de dado sensível (LGPD):       Coluna 'cpf' presente na tabela clientes")
print(f"
Taxa geral de No-Show na base: {agendamentos['no_show'].mean():.2%}")


In [ ]:
# 3. Análise Exploratória (Gráficos e Hipóteses Iniciais)
df_eda = agendamentos.merge(clientes, on='id_cliente').merge(veiculos, on='id_cliente')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Hipótese 1: Confirmação via WhatsApp reduz drasticamente o no-show
sns.barplot(data=df_eda, x='confirmou_msg', y='no_show', ax=axes[0], palette='Blues_d', errorbar=None)
axes[0].set_title('Taxa de No-Show por Confirmação (WhatsApp)')
axes[0].set_xticklabels(['Não Confirmou (0)', 'Confirmou (1)'])
axes[0].set_ylabel('Proporção de Faltas')

# Hipótese 2: Histórico de faltas anteriores aumenta o risco
sns.barplot(data=df_eda, x='faltas_anteriores', y='no_show', ax=axes[1], palette='Oranges_d', errorbar=None)
axes[1].set_title('Taxa de No-Show por Faltas Anteriores')
axes[1].set_xlabel('Número de Faltas Anteriores')
axes[1].set_ylabel('Proporção de Faltas')

# Hipótese 3: Agendamentos com mais de 10 dias de antecedência têm mais faltas
df_eda['antecedencia_alta'] = np.where(df_eda['dias_antecedencia'] > 10, '> 10 dias', '1 a 10 dias')
sns.barplot(data=df_eda, x='antecedencia_alta', y='no_show', ax=axes[2], palette='Greens_d', errorbar=None)
axes[2].set_title('Taxa de No-Show por Antecedência')
axes[2].set_xlabel('Antecedência do Agendamento')
axes[2].set_ylabel('Proporção de Faltas')

plt.tight_layout()
plt.show()


---
## Fase 3: Preparação dos Dados
**Objetivo Central:** Transformar, limpar e empacotar dados brutos em um formato digerível pelos algoritmos. Em projetos reais, esta etapa consome de **50% a 70%** do esforço total.

### Checklist de Execução
1. **Integrar (Mesclar):** Unir `agendamentos`, `clientes` e `veiculos` pela chave `id_cliente`.
2. **Selecionar (Filtrar):** Remover dados confidenciais (`cpf`) e identificadores sem valor preditivo (`id_agendamento`, `id_cliente`, `id_veiculo`).
3. **Limpar (Tratar):** Substituir `ano_veiculo > 2026` e `km_atual < 0` pela mediana válida; preencher nulos de `valor_estimado` com a mediana e `confirmou_msg` nulo com `0` (não confirmado).
4. **Construir (Derivar):** Criar o atributo derivado `idade_veiculo` ($2026 - 	ext{ano\_veiculo}$) e aplicar *One-Hot Encoding* nas variáveis categóricas (`tipo_servico`, `dia_semana`).


In [ ]:
# 1. INTEGRAR (Mesclar tabelas via ID)
df = agendamentos.merge(clientes, on='id_cliente').merge(veiculos, on='id_cliente')

# 2. SELECIONAR (Filtrar atributos confidenciais e IDs)
df = df.drop(columns=['cpf', 'id_agendamento', 'id_cliente', 'id_veiculo'])

# 3. LIMPAR (Corrigir erros de digitação e valores omissos)
mediana_ano = df.loc[df['ano_veiculo'] <= 2026, 'ano_veiculo'].median()
df.loc[df['ano_veiculo'] > 2026, 'ano_veiculo'] = mediana_ano

mediana_km = df.loc[df['km_atual'] >= 0, 'km_atual'].median()
df.loc[df['km_atual'] < 0, 'km_atual'] = mediana_km

df['valor_estimado'] = df['valor_estimado'].fillna(df['valor_estimado'].median())
df['confirmou_msg'] = df['confirmou_msg'].fillna(0)

# 4. CONSTRUIR (Derivar novos atributos e codificar categorias)
df['idade_veiculo'] = 2026 - df['ano_veiculo']
df = df.drop(columns=['ano_veiculo'])

df_prep = pd.get_dummies(df, columns=['tipo_servico', 'dia_semana'], drop_first=True)

# Verificação pós-preparação
print("=== VERIFICAÇÃO PÓS-PREPARAÇÃO (FASE 3) ===")
print(f"Dimensão final da base preparada: {df_prep.shape[0]} linhas x {df_prep.shape[1]} colunas")
print(f"Total de valores nulos restantes: {df_prep.isnull().sum().sum()}")
display(df_prep.head())


---
## Fase 4: Modelagem
**Objetivo Central:** Aplicar técnicas matemáticas, gerar designs de teste e iterar parâmetros para descobrir padrões de alto valor.

### Checklist de Execução
- **Selecionar técnicas:** Comparação entre **Regressão Logística** (linear, interpretável) e **Random Forest** (árvores de decisão, captura relações não lineares).
- **Gerar design de teste:** Partição estratificada em **80% treino** (800 registros) e **20% teste** (200 registros).
- **Iteração CRISP-DM (Não-Linearidade):** Ao preparar a Regressão Logística, o algoritmo exigiu variáveis numéricas na mesma escala. Retornamos à etapa de preparação para aplicar padronização (`StandardScaler`) exclusivamente ajustada no conjunto de treino (evitando vazamento de dados) e ajustamos o parâmetro `class_weight='balanced'` para melhorar a detecção da classe minoritária (`no_show = 1`).
- **Avaliação técnica estatística:** Acurácia, Precisão, Recall, F1-Score e ROC-AUC.


In [ ]:
# Separação em variáveis preditoras (X) e variável alvo (y)
X = df_prep.drop(columns=['no_show'])
y = df_prep['no_show']

# Design de Teste: Partição estratificada 80% Treino / 20% Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Iteração (Retorno à Preparação): Padronização exigida pela Regressão Logística
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo 1: Regressão Logística (com balanceamento de classes)
lr = LogisticRegression(random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

# Modelo 2: Random Forest (com balanceamento de classes)
rf = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]

# Tabela Comparativa de Métricas Estatísticas
resultados = pd.DataFrame({
    'Modelo': ['Regressão Logística', 'Random Forest'],
    'Acurácia': [accuracy_score(y_test, pred_lr), accuracy_score(y_test, pred_rf)],
    'Precisão (No-Show)': [precision_score(y_test, pred_lr), precision_score(y_test, pred_rf)],
    'Recall (No-Show)': [recall_score(y_test, pred_lr), recall_score(y_test, pred_rf)],
    'F1-Score': [f1_score(y_test, pred_lr), f1_score(y_test, pred_rf)],
    'ROC-AUC': [roc_auc_score(y_test, prob_lr), roc_auc_score(y_test, prob_rf)]
})

print("=== AVALIAÇÃO TÉCNICA ESTATÍSTICA (CONJUNTO DE TESTE: 200 REGISTROS) ===")
display(resultados.style.format({
    'Acurácia': '{:.2%}',
    'Precisão (No-Show)': '{:.2%}',
    'Recall (No-Show)': '{:.2%}',
    'F1-Score': '{:.2%}',
    'ROC-AUC': '{:.3f}'
}))


In [ ]:
# Visualização das Matrizes de Confusão e Curva ROC
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Matriz de Confusão - Regressão Logística
cm_lr = confusion_matrix(y_test, pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Prev: Veio', 'Prev: Faltou'],
            yticklabels=['Real: Veio', 'Real: Faltou'])
axes[0].set_title('Matriz de Confusão — Regressão Logística')

# Matriz de Confusão - Random Forest
cm_rf = confusion_matrix(y_test, pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Prev: Veio', 'Prev: Faltou'],
            yticklabels=['Real: Veio', 'Real: Faltou'])
axes[1].set_title('Matriz de Confusão — Random Forest')

# Curva ROC Comparativa
fpr_lr, tpr_lr, _ = roc_curve(y_test, prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, prob_rf)
axes[2].plot(fpr_lr, tpr_lr, label=f'Regressão Logística (AUC = {roc_auc_score(y_test, prob_lr):.3f})', linewidth=2)
axes[2].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, prob_rf):.3f})', linewidth=2)
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[2].set_title('Curva ROC Comparativa')
axes[2].set_xlabel('Taxa de Falsos Positivos')
axes[2].set_ylabel('Taxa de Verdadeiros Positivos (Recall)')
axes[2].legend(loc='lower right')

plt.tight_layout()
plt.show()


---
## Fase 5: Avaliação
**Objetivo Central:** Medir a eficácia dos modelos finais exclusivamente através da lente dos critérios de sucesso de negócio definidos na Fase 1 (**Impacto Real > Precisão Técnica**).

### Checklist de Execução
1. **Avaliar se os resultados respondem à meta:**
   - Embora ambos os modelos apresentem Acurácia e ROC-AUC muito semelhantes (~75% e ~0.80), a **Regressão Logística** obteve **Recall de 72,22%** na detecção de faltas (contra 61,11% do Random Forest).
   - Para o negócio da oficina, um **Falso Negativo** (prever que o cliente virá e ele faltar) custa um box parado. Um **Falso Positivo** custa apenas um lembrete automático no WhatsApp. Logo, a Regressão Logística gera maior impacto real.
2. **Revisar falhas e limitações:**
   - Como a base é sintética (1.000 registros), variáveis externas reais (como chuvas intensas, trânsito ou vésperas de feriado) ainda não estão presentes.
3. **Decisão (Implementar ou Iterar?):**
   - **Aprovado para implementação piloto**, prevendo uma nova iteração à Fase 2 assim que a TechMotors acumular histórico real em produção.


In [ ]:
# Simulação de Impacto Real de Negócio (Critério da Fase 1: Reduzir no-show em >= 20%)
total_faltas_reais = (y_test == 1).sum()
faltas_detectadas_lr = cm_lr[1, 1]

# Assumindo que ações preventivas (contato ativo/remanejamento) recuperam 40% das faltas detectadas
taxa_recuperacao_acao = 0.40
faltas_evitadas = faltas_detectadas_lr * taxa_recuperacao_acao
reducao_percentual_noshow = faltas_evitadas / total_faltas_reais

print("=== SIMULAÇÃO DE IMPACTO DE NEGÓCIO (FASE 5) ===")
print(f"Faltas reais no período de teste (200 agendamentos): {total_faltas_reais}")
print(f"Faltas previstas antecipadamente (Regressão Logística): {faltas_detectadas_lr} ({faltas_detectadas_lr/total_faltas_reais:.2%} de Recall)")
print(f"Estimativa de faltas evitadas com ação preventiva (40% conversão): {faltas_evitadas:.1f} agendamentos")
print(f"Redução estimada na taxa de No-Show: {reducao_percentual_noshow:.2%}")
print(f"Meta de Negócio da Fase 1 (>= 20%): {'ATINGIDA ✅' if reducao_percentual_noshow >= 0.20 else 'NÃO ATINGIDA ❌'}")


---
## Fase 6: Implementação
**Objetivo Central:** Incorporar formalmente o modelo ao sistema da TechMotors ou utilizar as descobertas para orientar decisões estratégicas.

### Entregáveis Finais
1. **Plano de Implementação (Arquitetura):**
   - O modelo vencedor (`Regressão Logística`) e o padronizador (`StandardScaler`) são serializados em arquivos `.joblib` e expostos via API REST (Python/FastAPI).
   - O backend da TechMotors consulta a API 48h antes de cada agendamento: clientes classificados com risco **Alto** ($P \ge 0.65$) recebem contato ativo/confirmação prioritária via WhatsApp.
2. **Plano de Monitoramento (Obsolescência do Modelo):**
   - Acompanhamento mensal comparando o *no-show* previsto com o realizado. Quedas de Recall superiores a 5 pontos percentuais disparam retreinamento automático com dados reais atualizados.
3. **Relatório e Apresentação Final:**
   - Este notebook + gráficos gerados servem como evidência técnica e gerencial para a equipe de TI e para o TCC.
4. **Revisão do Projeto (Lições Aprendidas):**
   - A validação de dados (ex.: impedir anos de veículo futuros e quilometragem negativa) deve ser aplicada diretamente no formulário web da TechMotors, reduzindo drasticamente o esforço futuro da Fase 3.


In [ ]:
# Exportação dos artefatos do modelo para produção (Simulação de Deploy)
joblib.dump(lr, 'modelo_noshow_techmotors.joblib')
joblib.dump(scaler, 'scaler_techmotors.joblib')

# Função de inferência simulando uma chamada da API da TechMotors para um novo agendamento
def prever_risco_noshow(dados_agendamento_dict):
    novo_df = pd.DataFrame([dados_agendamento_dict])
    # Alinha as colunas com o formato exato esperado pelo modelo treinado
    novo_df = novo_df.reindex(columns=X.columns, fill_value=0)
    novo_scaled = scaler.transform(novo_df)
    prob = lr.predict_proba(novo_scaled)[0, 1]
    
    if prob >= 0.65:
        nivel = "ALTO RISCO -> Disparar confirmação ativa via WhatsApp"
    elif prob >= 0.40:
        nivel = "MÉDIO RISCO -> Enviar lembrete padrão 24h antes"
    else:
        nivel = "BAIXO RISCO -> Fluxo normal"
    return prob, nivel

# Teste prático com um agendamento fictício de alto risco
exemplo_cliente = {
    'dias_antecedencia': 15,
    'valor_estimado': 450.0,
    'confirmou_msg': 0,
    'idade': 29,
    'faltas_anteriores': 2,
    'idade_veiculo': 8,
    'tipo_servico_Revisão Geral': 1,
    'dia_semana_Segunda': 1
}

prob_exemplo, acao = prever_risco_noshow(exemplo_cliente)
print("=== TESTE DE INFERÊNCIA EM PRODUÇÃO (FASE 6) ===")
print("Artefatos exportados: 'modelo_noshow_techmotors.joblib' e 'scaler_techmotors.joblib'")
print(f"Probabilidade prevista de No-Show: {prob_exemplo:.2%}")
print(f"Ação recomendada pelo sistema:     {acao}")
